# 🚢 Maritime VHF ASR — Dissertation Notebook
**Transformer-Based ASR Fine-Tuning on Real and Simulated Maritime Speech**

---
### Before you start
1. **Runtime → Change runtime type → T4 GPU**
2. Have your **Label Studio API key** ready → https://app.heartex.com/user/account

| Model | GPU needed |
|---|---|
| whisper-small / wav2vec2 | T4 (free tier) |
| whisper-medium | T4 (~30 min / experiment) |
| whisper-large + LoRA | A100 (Pro+) |

## ✅ Step 1 — Check GPU & Mount Drive

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"✅  GPU : {gpu.name}")
    print(f"   VRAM: {gpu.total_memory/1e9:.1f} GB")
else:
    print("❌  No GPU — Runtime → Change runtime type → T4 GPU")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

## 📁 Step 2 — Sync Project from GitHub

**⚠️ IMPORTANT: Run this cell at the start of every new Colab session.**

**Also run it again whenever:**
- You've pushed code changes to GitHub from your local machine
- You want to pull the latest updates from the repository

**What this cell does:**
- First time: Clones the repository from GitHub
- Subsequent times: Pulls latest changes with `git pull`

**💡 Note:** If `Maritime_ASR_Colab.ipynb` itself has been updated on GitHub, you'll need to:
1. Run this cell to pull the updated notebook
2. Go to **File → Open notebook → `/content/asr_dissertation/Maritime_ASR_Colab.ipynb`**
3. The updated version will open in a new tab
4. Close the old tab and continue working in the new one

In [ ]:
import os

REPO = "https://github.com/tolgaSarmi/maritime_asr.git"
WORKDIR = "/content/asr_dissertation"

if os.path.exists(WORKDIR):
    print("📥 Pulling latest changes from GitHub...")
    !git -C {WORKDIR} pull
    print("✅ Code updated from GitHub")
else:
    print("📦 Cloning repository from GitHub...")
    !git clone {REPO} {WORKDIR}
    print("✅ Repository cloned")

os.chdir(WORKDIR)
print(f"📂 Working directory: {os.getcwd()}")

### 📌 Quick Reference: GitHub Workflow

**On your local machine (Mac/PC):**
```bash
cd ~/Desktop/asr_dissertation_with_colab_1
# Make your code changes, then:
git add -A
git commit -m "Your commit message"
git push
```

**In Colab:**
- Run the cell above (Step 2) to pull the changes
- If the notebook itself changed, reopen it from the new location

**Setting up Google Drive persistence:**

The project clones to `/content/asr_dissertation` (temporary).

Checkpoints, results, and manifests are saved to Google Drive at:
`/content/drive/MyDrive/ASR_Dissertation/`

## 📦 Step 3 — Install Dependencies

Run **Cell A**, then **Runtime → Restart session**, then run **Cell B** and **Cell C**.
Do NOT re-run Cell A after restarting.


In [ ]:
# ── Cell A: install everything in one shot, then restart the session ──
!pip install -q \
    "scipy>=1.14.0" \
    "scikit-learn>=1.6.0" \
    transformers peft datasets accelerate evaluate \
    jiwer librosa soundfile omegaconf rich inflect \
    tensorboard seaborn
print("✅  Done — Runtime → Restart session now")


In [ ]:
# ── Cell B: run immediately after restart ──
import os
os.chdir('/content/asr_dissertation')
print("📂  Working directory:", os.getcwd())


In [ ]:
# ── Cell C: verify imports ──
import numpy, scipy, transformers, peft, omegaconf, evaluate
print(f"numpy        : {numpy.__version__}")
print(f"scipy        : {scipy.__version__}")       # must be 1.14+
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print("✅  All imports OK")


## ⚠️ Step 4 — Code Patches (No Longer Required)

**All bug fixes are now permanently integrated into source control.**

Previous versions of this notebook applied runtime patches to fix:
- `export_tasks()` pagination issues
- Azure SAS authentication for audio downloads
- Whisper feature collation errors

✅ **These fixes are now in the repository** — no patches needed!

The GitHub sync (Step 2) automatically pulls the corrected versions:
- `label_studio_export.py` (with robust pagination + plain requests)
- `src/dataset.py` (with correct tensor stacking)

**You can skip directly to Step 5 (API Key & Download Data).**

## 🔑 Step 5 — Label Studio API Key & Download Data

**Real dataset** → audio streams from Azure during training (no download)

**Simulated dataset** → audio downloaded to disk (public Azure container)


In [ ]:
import os
try:
    from google.colab import userdata
    api_key = userdata.get('LABEL_STUDIO_API_KEY')
    print('✅  API key loaded from Colab Secrets')
except Exception:
    api_key = ''   # ← paste key here if not using Secrets
    if api_key: print('✅  API key set manually')
    else: print('⚠️   No API key — add to Colab Secrets as LABEL_STUDIO_API_KEY')

os.environ['LABEL_STUDIO_API_KEY'] = api_key


In [ ]:
# ── Real dataset: fetch metadata + URLs (no audio download) ──
# Heartex cloud uses 100 tasks/page (API limit)
!python label_studio_export.py --api-key $LABEL_STUDIO_API_KEY --project real
# Expected: ✅ [real] Exported=1556  Failed=0

# ── Simulated dataset: download audio to disk ──
!python label_studio_export.py --api-key $LABEL_STUDIO_API_KEY --project simulated
# Expected: ✅ [simulated] Exported=1916  Failed=0

In [ ]:
# ── RESTORE MANIFESTS FROM DRIVE (run instead of Step 5 in future sessions) ──
# Copies manifests from Drive — skips the Label Studio API export entirely
DRIVE_PROJECT = '/content/drive/MyDrive/ASR_Dissertation'
import shutil, os, glob

manifests = glob.glob(f'{DRIVE_PROJECT}/data/**/*.json', recursive=True)
if not manifests:
    print('⚠️  No saved manifests found — run Step 5 export first')
else:
    for src in manifests:
        rel = os.path.relpath(src, f'{DRIVE_PROJECT}/')
        os.makedirs(os.path.dirname(rel), exist_ok=True)
        shutil.copy(src, rel)
        print(f'Restored: {rel}')
    print('✅  Manifests restored — skip Step 5, go directly to Step 6')


In [ ]:
# ── SAVE MANIFESTS TO DRIVE (run once after first successful export) ──
# Saves the JSON manifests so future sessions skip the Label Studio API calls
DRIVE_PROJECT = '/content/drive/MyDrive/ASR_Dissertation'
import shutil, os, glob

os.makedirs(f'{DRIVE_PROJECT}/data', exist_ok=True)
for manifest in glob.glob('data/**/*.json', recursive=True):
    dst = f'{DRIVE_PROJECT}/{manifest}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(manifest, dst)
    print(f'Saved: {manifest}')
print('✅  Manifests saved to Drive — future sessions can restore instead of re-exporting')


## 🔧 Step 6 — Prepare Datasets

In [ ]:
# Validate audio, normalise transcriptions, create 80/10/10 train/val/test splits
!python main.py --mode data
# Expected: train ~1244 | val ~155 | test ~157


In [ ]:
import json, glob
for path in sorted(glob.glob('data/*/*.json')):
    with open(path) as f:
        n = len(json.load(f))
    print(f"{path:<50} {n:>5} samples")


## 🏋️ Step 7 — Training

**Start with this to verify your setup works (~30 min on T4):**
`ef_whisper_small_real`

Then run the full batch cell below.


In [ ]:
!python main.py --mode list

In [ ]:
# ── Single experiment ──
EXPERIMENT = "ef_whisper_small_real"
!python main.py --mode train --experiment {EXPERIMENT}


### Run all real-data experiments

In [ ]:
# Runs sequentially; already-finished experiments are skipped automatically.
# After a session reset: restore checkpoints (Step 8 RESTORE), then re-run.

EXPERIMENTS = [
    # Encoder freezing
    "ef_whisper_small_real",
    "ef_whisper_medium_real",
    "ef_wav2vec2_real",
    # LoRA (fixed: expanded target_modules, LR=1e-3, no 8-bit quant)
    "lora_whisper_small_real",
    "lora_whisper_medium_real",
    # ── Uncomment for A100 (Pro+) ──────────────────────────────────────
    # "ef_whisper_large_real",
    # "lora_whisper_large_real",
    # "full_ft_whisper_medium_real",
    # "full_ft_whisper_large_real",
]

import subprocess, sys
for exp in EXPERIMENTS:
    print(f"\n{'='*55}\n  {exp}\n{'='*55}")
    r = subprocess.run(
        [sys.executable, "main.py", "--mode", "train", "--experiment", exp],
        cwd="/content/asr_dissertation",
    )
    if r.returncode != 0:
        print(f"  ⚠️  {exp} failed — continuing...")


## 💾 Step 8 — Save / Restore Checkpoints from Drive

In [ ]:
DRIVE_PROJECT = '/content/drive/MyDrive/ASR_Dissertation'
# SAVE — run after each training session
import shutil, os

src = '/content/asr_dissertation/checkpoints'
dst = f'{DRIVE_PROJECT}/checkpoints'
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"✅  Checkpoints saved to {dst}")
else:
    print("No checkpoints yet — run training first")


In [ ]:
DRIVE_PROJECT = '/content/drive/MyDrive/ASR_Dissertation'
# RESTORE — run at the start of a new session (after Step 3 Cell B)
import shutil

src = f'{DRIVE_PROJECT}/checkpoints'
dst = '/content/asr_dissertation/checkpoints'
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("✅  Checkpoints restored from Drive")
else:
    print("No saved checkpoints on Drive yet")


## 📊 Step 9 — Evaluate All Experiments

In [ ]:
!python main.py --mode eval_all

In [ ]:
# Evaluate a specific checkpoint on the test set
CHECKPOINT = "checkpoints/ef_whisper_small_real"
MANIFEST   = "data/real/test_manifest.json"
!python main.py --mode test_wer --checkpoint {CHECKPOINT} --manifest {MANIFEST}


## 📈 Step 10 — Generate Dissertation Figures

In [ ]:
!python main.py --mode figures

import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f)


In [ ]:
from IPython.display import Image, display
import glob

for fig in sorted(glob.glob('results/figures/*.png')):
    print(f"\n── {fig.split('/')[-1]} ──")
    display(Image(fig, width=900))


In [ ]:
DRIVE_PROJECT = '/content/drive/MyDrive/ASR_Dissertation'
import shutil
shutil.copytree('/content/asr_dissertation/results',
                f'{DRIVE_PROJECT}/results', dirs_exist_ok=True)
print(f"✅  Results saved to Drive")


## 📋 Step 11 — Results Summary Table

In [ ]:
import json, pandas as pd

try:
    with open('results/all_results.json') as f:
        all_results = json.load(f)

    rows = []
    for exp_name, res in all_results.items():
        for key, val in res.items():
            if key.startswith('eval_') and isinstance(val, dict) and 'wer' in val:
                rows.append({
                    'Experiment': exp_name,
                    'Method'    : res.get('method', ''),
                    'Train Data': str(res.get('train_data', '—')),
                    'Eval Set'  : key.replace('eval_', ''),
                    'WER (%)'   : round(val['wer'] * 100, 2),
                    'CER (%)'   : round(val.get('cer', 0) * 100, 2),
                })

    df = pd.DataFrame(rows).sort_values(['Method', 'Eval Set', 'WER (%)'])
    pd.set_option('display.max_rows', 100)
    print(df.to_string(index=False))

except FileNotFoundError:
    print("No results yet — run: python main.py --mode eval_all")


## 🚧 Step 12 — Simulated Data (when Azure access is restored)

The `sim_vhf_dataset` SAS tokens expired Feb 2026 and the Azure Blob Storage
connection was removed from Label Studio.

**To re-enable:**
1. Email **Derek Flanagan** (flanagan.derek@ul.ie) — ask him to reconnect
   `simvhf.blob.core.windows.net` to Label Studio project 226738
2. Once reconnected, update Step 5:
   ```
   --project both
   ```
3. Re-run Steps 6–11 to add the simulated / combined experiments:
   - `ef_whisper_small_simulated` / `ef_whisper_small_combined`
   - `lora_whisper_small_simulated` / `lora_whisper_small_combined`
   - etc.


## 🎙️ Bonus — Transcribe a Single File

In [ ]:
from src.inference import build_transcriber
from google.colab import files

print("Upload an audio file (WAV / MP3)...")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

CHECKPOINT = "checkpoints/ef_whisper_small_real"   # ← change to best model

transcriber = build_transcriber("whisper", CHECKPOINT)
text, elapsed = transcriber.transcribe_file(audio_path)

print(f"\n📝  Transcription : {text}")
print(f"⏱️   Latency        : {elapsed*1000:.0f} ms")


---
## 🔄 Session Reset Checklist

After Colab disconnects or runtime restarts:

1. ✅ Step 1 — GPU check
2. 📁 Step 2 — mount Drive, re-upload zip
3. 📦 Step 3 — Cell A → restart → Cell B → Cell C
4. 🔧 **Step 4 — apply all 3 patches** (required every session)
5. ⚡ **Step 5 RESTORE** — restore manifests from Drive (~2 sec, skip full export)
6. 🔧 Step 6 — re-run data pipeline (~30s)
6. 💾 **Step 8 RESTORE** — pull checkpoints from Drive
7. 🏋️ Step 7 / Step 9 — continue training or go straight to eval

---
## ⚠️ Out of Memory?

Reduce batch size in `configs/config.yaml`:
```yaml
per_device_train_batch_size: 8   # reduce from 16
gradient_accumulation_steps: 4   # keep effective batch = 32
dataloader_num_workers: 2        # reduce from 4
fp16: true                       # already enabled for T4
```
